# Пари 41–42 · RAG та AI-агент · 180 хв

40 хв — підготовка API і повторення промптів; 70 хв — RAG; 70 хв — Function Calling, агент та інтеграція.

## Налаштування API

40 хв — промпти й перевірка відповідей; 70 хв — RAG; 70 хв — агент та інтеграція.
Працюйте у цій папці. Потрібен Python 3.10+.

## Налаштування

```bash
python -m venv .venv
# Windows PowerShell:
.venv\Scripts\Activate.ps1
# macOS/Linux: source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows: `Copy-Item .env.example .env`. macOS/Linux: `cp .env.example .env`.
Відкрийте `.env` локально та вставте власний ключ у OPENAI_API_KEY.
OPENAI_MODEL=gpt-4.1-mini; якщо проєкт не має доступу, впишіть доступну вам модель.
Ключ створюється на https://platform.openai.com/api-keys.
API має окремий білінг; підписка ChatGPT не надає автоматично API-кредитів.
Кожен запуск команди надсилає реальний запит і може коштувати грошей.
Встановіть бюджет у проєкті; не публікуйте ключ, .env чи приватні відповіді.

```bash
python api_client.py requests/task_01.json --save results/task_01.json
```

Відредагуйте input та instructions у JSON — і повторіть команду.
Клієнт містить тільки транспорт, читання .env та діагностику. Розв’язки
класифікатора, пошуку і агента ви пишете самі у student_work.py.
Запити 09–10 потребують вашого tools: схеми get_schedule, а потім get_average.
Ліміти: timeout 30 секунд, без автоматичних повторів, до 2000 вихідних токенів на запит.
Це ліміт виводу, а не гарантований бюджет вартості. Статус incomplete означає,
що відповідь могла обірватися: перевірте status перед використанням JSON.

401 — перевірте ключ; 429 — квоту й частоту; 404/model_not_found — доступ до моделі.
Не друкуйте ключ під час діагностики. Не зберігайте його в HTML або браузерному JS.
Результати в results/ залишаються локально. Записуйте лише необхідні навчальні дані.

## Коротка карта

1. Промпт задає завдання; приклади показують бажаний формат.
2. Контекст містить дані для поточного запиту; few-shot не навчає ваги.
3. JSON від моделі потребує програмної перевірки.
4. RAG: питання → пошук → контекст → модель → перевірка джерел.
5. TF-IDF шукає збіги слів; це не семантичні embeddings.
6. Function Calling: модель пропонує інструмент → Python перевіряє → виконує → повертає observation.
7. Агент повторює цей цикл із лімітом кроків.

Спочатку обов’язкова практика. Додатково порівняйте Critic/Editor з власним
валідатором, Planner–Executor з агентним циклом; не вимагайте прихованих
міркувань моделі — просіть коротке пояснення та перевірювані проміжні результати.
Google AI Studio/Vertex AI — окремі платформи; цей практикум працює через OpenAI.

## Що здати

student_work.py, власні JSON-запити, 4–6 тестів без мережі, короткий REPORT.md.
У REPORT: версія промпту, питання, очікуване, отримане, джерела, model/status/usage,
ваш висновок. Для zero/few-shot потрібно 10 відповідей; решта мінімум по картках.
Не підміняйте реальні відповіді заздалегідь написаними fixtures.
Оцінювання: промпти й валідація — 20; пошук та RAG — 30; агент — 30;
тести, відтворюваність, пояснення обмежень — 20 балів.

Офіційна документація: https://developers.openai.com/api/reference/python
та https://developers.openai.com/api/docs/guides/function-calling


## 05. Власний корпус і чанки · 15 хв

Збережіть походження кожного фрагмента.

1. Додайте четвертий документ про broadcasting у data/documents.json.
2. Реалізуйте chunks у student_work.py: вікно 40 слів, overlap 10, id документа, початкова позиція.
3. Порівняйте 20/5 і 40/10. Один фрагмент передайте моделі як input через JSON-файл.

`python api_client.py requests/task_05.json --save results/task_05.json`

Підказка: Тут слова, не токени. Не називайте кількість слів точним розміром контексту моделі.

Критерій: ID унікальні; текст і source не загублені; overlap >= size відхиляється; є жива відповідь за вашим фрагментом.


## 06. TF-IDF пошук top-k · 20 хв

Знайдіть релевантний контекст до генерації.

1. Реалізуйте retrieve через TfidfVectorizer і cosine_similarity.
2. Знайдіть top-2 для «python параметри»; виведіть ID та score.
3. Перевірте порожній корпус, повторні ID й невідоме слово. Знайдені фрагменти вручну передайте API.

`python api_client.py requests/task_06.json --save results/task_06.json`

Підказка: fit робіть на корпусі, transform — на питанні. TF-IDF є лексичним пошуком, не embeddings.

Критерій: Нульові score відкинуті; невідоме слово дає []; модель отримала тільки знайдені вами фрагменти.


## 07. Повний RAG із живою генерацією · 20 хв

З’єднайте власний пошук і реальний OpenAI API.

1. Реалізуйте rag: питання → chunks → retrieve → input з фрагментами → send.
2. Попросіть JSON answer/sources, перевірте типи полів і належність ID знайденим фрагментам.
3. Якщо контекст порожній, поверніть відмову локально без API-запиту. Виконайте два питання з контекстом.

`python api_client.py requests/task_07.json --save results/task_07.json`

Підказка: send імпортуйте з api_client. Відповідь доступна як response.output_text. Підтримуйте JSONDecodeError.

Критерій: Два живі результати з перевіреними джерелами; негативний кейс не викликає API; кожне твердження вручну звірено з текстом.


## 08. Оцінювання RAG і prompt injection · 15 хв

Перевірте пошук і генерацію окремо.

1. Складіть 4 питання та еталонні ID; порахуйте recall@2: частка еталонних джерел серед top-2, потім середнє за питаннями.
2. Додайте документ із фразою «ігноруй інструкції» та перевірте відповідь API.
3. Порівняйте два розміри чанків; запишіть якість пошуку, підтримку фактів і токени окремо.

`python api_client.py requests/task_08.json --save results/task_08.json`

Підказка: Інструкція всередині документа є недовіреними даними. Результат одного експерименту не доводить повний захист.

Критерій: Таблиця із чотирма запитами; один живий adversarial-запит; немає твердження, що валідний ID доводить правдивість.


## 09. Function Calling: отримайте справжній виклик · 15 хв

Побачте function_call у відповіді моделі.

1. Опишіть у requests/task_09.json інструмент get_schedule: course — рядок, required, additionalProperties=false, strict=true.
2. Надішліть питання «Коли Python?» і збережіть повний JSON через --save.
3. Знайдіть у output name, arguments, call_id. Реалізуйте execute з allowlist, без eval і shell.

`python api_client.py requests/task_09.json --save results/task_09.json`

Підказка: Схему інструмента допишіть самі у поле tools. Модель пропонує виклик; Python виконує його.

Критерій: Є живий function_call; невідомий інструмент, зайвий параметр і не-рядок відхиляються локально.


## 10. Агентний цикл · 25 хв

Поверніть результат інструмента моделі та отримайте фінальний текст.

1. Реалізуйте agent: запит до API → output → function_call → перевірка аргументів → execute.
2. Додайте весь response.output до історії та function_call_output з тим самим call_id і JSON-рядком output; надішліть наступний запит.
3. Обмежте цикл трьома API-запитами; додайте get_average з перевіркою чисел 0–100 і власний тест.

`python api_client.py requests/task_10.json --save results/task_10.json`

Підказка: Клієнт приймає input-масив і tools. Повну історію надсилайте повторно; store=false, тому не покладайтеся на серверне збереження.

Критерій: Живий цикл закінчується текстовою відповіддю; tool output прив’язаний до call_id; ліміт зупиняє цикл; API не викликається під час import.


## 11. Інтеграція та захист · 30 хв

Покажіть працездатний помічник із власним кодом.

1. Додайте CLI: режим prompt/rag/agent і власне питання. Виконайте щонайменше один живий сценарій у кожному режимі.
2. Перевірте неправильний ключ, недоступну модель і квоту за таблицею помилок; не витрачайте запити навмисно — для автоматичного тесту використайте mock.
3. Здайте код, 4–6 локальних тестів, журнал дослідів і короткий опис обмежень. За наявності часу додайте FastAPI /health та /ask; ключ лише на сервері.

`python api_client.py requests/task_11.json --save results/task_11.json`

Підказка: Текст відповіді недетермінований: перевіряйте контракт і джерела, а не точну фразу для довільної генерації.

Критерій: Інша людина копіює .env.example, встановлює залежності й запускає ваш код; .env не потрапив у Git; результати містять status/model/usage.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "PythonAI_Author_Lesson41_Lesson42"))

"""Ваші реалізації; імпорт цього файлу не надсилає API-запитів."""
from api_client import send


def classify(texts, ask=send):
    """Завдання 02: перевірені мітки для відгуків."""
    raise NotImplementedError("Реалізуйте за карткою завдання")


def chunks(documents, size=40, overlap=10):
    """Завдання 05: фрагменти з походженням."""
    raise NotImplementedError("Реалізуйте за карткою завдання")


def retrieve(question, documents, k=2):
    """Завдання 06: top-k із додатною схожістю."""
    raise NotImplementedError("Реалізуйте за карткою завдання")


def rag(question, documents, ask=send):
    """Завдання 07: відповідь і перевірені джерела."""
    raise NotImplementedError("Реалізуйте за карткою завдання")


def execute(name, arguments):
    """Завдання 09: allowlist і перевірка параметрів."""
    raise NotImplementedError("Реалізуйте за карткою завдання")


def agent(question, ask=send, max_steps=3):
    """Завдання 10: обмежений цикл інструментів."""
    raise NotImplementedError("Реалізуйте за карткою завдання")
